# Memora AI - Code Error Analysis & Model Training Pipeline

This notebook trains the local machine learning model using **`tests/data.json`** as the dataset, persisting verified training examples directly into the **Memora AI Local SQLite Database** (`memoradb.db`).

In [ ]:
# 1. Setup & Imports
import os
import sys
import json
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Ensure backend root is in python path
sys.path.insert(0, str(Path.cwd()))

from app.db.session import SessionLocal, engine, Base
from app.db.models import User, ErrorClassificationExample, CodeSubmission
from app.training.dataset import DATA_JSON_PATH, ingest_data_json_into_db, load_verified_training_examples
from app.training.train_model import MODEL_PATH, train_error_classifier
from app.training.predict import predict_error_type

print("✓ Imports successful.")
print(f"✓ Dataset file path: {DATA_JSON_PATH} (Exists: {DATA_JSON_PATH.exists()})")

## 2. Ingest `tests/data.json` into Local Database (`memoradb.db`)

Ingests verified code defect samples from `tests/data.json` into the `error_classification_examples` table in the local database.

In [ ]:
Base.metadata.create_all(bind=engine)
db = SessionLocal()

# Ingest from tests/data.json
new_ingested = ingest_data_json_into_db(db, max_records=500)
total_in_db = db.query(ErrorClassificationExample).filter_by(is_verified=True).count()

print(f"✓ Ingested {new_ingested} new samples from tests/data.json.")
print(f"✓ Total verified training records in local database: {total_in_db}")

## 3. Load Dataset into Pandas DataFrame & Exploratory Data Analysis (EDA)

In [ ]:
records = db.query(ErrorClassificationExample).filter(ErrorClassificationExample.is_verified.is_(True)).all()

data = []
for r in records:
    combined_text = f"{r.user_code}\n{r.stack_trace or ''}\n{r.pattern_cluster or ''}".strip()
    data.append({
        "id": r.id,
        "language": r.programming_language,
        "code": r.user_code[:200] + "..." if len(r.user_code) > 200 else r.user_code,
        "pattern_cluster": r.pattern_cluster,
        "error_type": r.error_type,
        "text": combined_text
    })

df = pd.DataFrame(data)
print(f"✓ Loaded {len(df)} samples into Pandas DataFrame.\n")

print("=== Error Type Distribution from tests/data.json ===")
print(df["error_type"].value_counts())

print("\n=== Programming Language Breakdown ===")
print(df["language"].value_counts())

df.head()

## 4. Feature Extraction & Model Training Pipeline

Train a **TF-IDF Vectorizer + Multinomial Logistic Regression** model on the dataset.

In [ ]:
X = df["text"]
y = df["error_type"]

# Define end-to-end Machine Learning Pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=10000)),
    ("clf", LogisticRegression(max_iter=1000, solver="lbfgs"))
])

# Fit model on local dataset
pipeline.fit(X, y)
train_preds = pipeline.predict(X)
acc = accuracy_score(y, train_preds)

print(f"✓ Model fitted on {len(df)} samples across {len(pipeline.classes_)} classes.")
print(f"✓ Training Accuracy: {acc * 100:.2f}%")
print("\n=== Classes Learned ===")
for c in pipeline.classes_:
    print(f"  - {c}")

## 5. Detailed Classification Metrics

In [ ]:
print("=== Detailed Classification Report ===")
print(classification_report(y, train_preds, zero_division=0))

## 6. Export Model Artifact to Backend Directory

Save the trained model pipeline into `app/training/artifacts/error_classifier.pkl` so the live FastAPI backend uses the updated weights.

In [ ]:
target_artifact_path = Path(MODEL_PATH)
target_artifact_path.parent.mkdir(parents=True, exist_ok=True)

with open(target_artifact_path, "wb") as f:
    pickle.dump(pipeline, f)

print(f"✓ Model artifact successfully serialized to: {target_artifact_path}")

## 7. Real-time Inference Verification

In [ ]:
test_samples = [
    ("int* ptr = NULL; *ptr = 100;", "C Null Pointer Dereference"),
    ("arr = [1, 2, 3]; print(arr[50])", "Python Index Out of Range"),
    ("fp = open('output.txt', 'r'); lines = fp.readlines()", "Python File Handle / Resource Leak"),
    ("def binary_search(arr, target): mid = (left + right) // 2", "Python Binary Search Invariants"),
    ("def infinite(): return infinite()", "Python Infinite Recursion")
]

print("=== Real-time Model Predictions ===")
for code_snippet, label in test_samples:
    prediction = predict_error_type(code_snippet, db)
    print(f"[{label}]\nSnippet: {code_snippet}\n-> Predicted Defect: {prediction}\n")

db.close()
print("✓ Database connection closed. Model pipeline execution complete.")